# RWGNN Inference & Training Demo

This notebook demonstrates how to generate small XY-model lattices, train the Random Walk GNN on-the-fly, and visualise predictions for BLT phase classification and critical-temperature estimation. The workflow mirrors the `RWGNN/train.py` pipeline but keeps the runtime modest for interactive exploration.


In [ ]:
import json
from pathlib import Path
import math
import random
import numpy as np
import torch
from torch_geometric.loader import DataLoader
import matplotlib.pyplot as plt

from RWGNN.data import generate_from_simulator
from RWGNN.model import RandomWalkGNN

# Ensure deterministic-ish runs for the quick demo
random.seed(7)
np.random.seed(7)
torch.manual_seed(7)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device


In [ ]:
# Configuration for a compact demo run
lattice_shape = (8, 8)  # keeps graphs small for fast training
critical_temperature = 0.89
coupling = 1.0
steps = 30
iters_per_step = 20

# Sample a few temperatures across the ordered/disordered regimes
train_temps = np.linspace(0.6, 1.2, 7)
val_temps = np.array([0.75, 0.95, 1.1])
test_temps = np.array([0.7, 0.9, 1.05, 1.2])

samples_per_temp = 3  # keeps runtime low
walk_length = 4
num_walks = 8
batch_size = 8
hidden_channels = 96
dropout = 0.1
epochs = 6
learning_rate = 3e-3

config = {
    'lattice_shape': lattice_shape,
    'critical_temperature': critical_temperature,
    'train_temps': train_temps.tolist(),
    'val_temps': val_temps.tolist(),
    'test_temps': test_temps.tolist(),
    'samples_per_temp': samples_per_temp,
    'walk_length': walk_length,
    'num_walks': num_walks,
    'batch_size': batch_size,
    'hidden_channels': hidden_channels,
    'dropout': dropout,
    'epochs': epochs,
    'learning_rate': learning_rate,
}
config


In [ ]:
# Generate graphs directly from the XY model simulator
train_graphs = generate_from_simulator(
    train_temps, samples_per_temp,
    lattice_shape=lattice_shape,
    steps=steps,
    iters_per_step=iters_per_step,
    seed=42,
    coupling=coupling,
    walk_length=walk_length,
    num_walks=num_walks,
    critical_temperature=critical_temperature,
)
val_graphs = generate_from_simulator(
    val_temps, samples_per_temp,
    lattice_shape=lattice_shape,
    steps=steps,
    iters_per_step=iters_per_step,
    seed=314,
    coupling=coupling,
    walk_length=walk_length,
    num_walks=num_walks,
    critical_temperature=critical_temperature,
)
test_graphs = generate_from_simulator(
    test_temps, samples_per_temp,
    lattice_shape=lattice_shape,
    steps=steps,
    iters_per_step=iters_per_step,
    seed=2718,
    coupling=coupling,
    walk_length=walk_length,
    num_walks=num_walks,
    critical_temperature=critical_temperature,
)

len(train_graphs), len(val_graphs), len(test_graphs)


In [ ]:
# Wrap into loaders
train_loader = DataLoader(train_graphs, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_graphs, batch_size=batch_size)
test_loader = DataLoader(test_graphs, batch_size=batch_size)

sample = train_graphs[0]
in_channels = sample.x.shape[1]
in_channels


In [ ]:
model = RandomWalkGNN(in_channels=in_channels, hidden_channels=hidden_channels, dropout=dropout).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

model


In [ ]:
def run_epoch(loader, *, training: bool):
    if training:
        model.train()
    else:
        model.eval()
    total_loss = 0.0
    total_cls = 0.0
    total_samples = 0
    correct_phase = 0
    with torch.set_grad_enabled(training):
        for batch in loader:
            batch = batch.to(device)
            if training:
                optimizer.zero_grad()
            outputs = model(batch)
            loss, parts = model.compute_losses(
                outputs,
                phase_target=batch.y,
                temp_target=batch.temperature,
                tc_target=batch.tc_target,
            )
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * batch.num_graphs
            total_cls += parts['phase_ce'].item() * batch.num_graphs
            preds = outputs['phase_logits'].argmax(dim=-1)
            correct_phase += (preds == batch.y.view(-1)).sum().item()
            total_samples += batch.num_graphs
    return {
        'loss': total_loss / total_samples,
        'phase_loss': total_cls / total_samples,
        'phase_acc': correct_phase / total_samples,
    }

history = {'train': [], 'val': []}
for epoch in range(1, epochs + 1):
    train_stats = run_epoch(train_loader, training=True)
    val_stats = run_epoch(val_loader, training=False)
    history['train'].append(train_stats)
    history['val'].append(val_stats)
    print(f"Epoch {epoch:02d} | train loss={train_stats['loss']:.3f} acc={train_stats['phase_acc']:.2f} | val acc={val_stats['phase_acc']:.2f}")


In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot([m['loss'] for m in history['train']], label='train')
axes[0].plot([m['loss'] for m in history['val']], label='val')
axes[0].set_title('Total loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot([m['phase_acc'] for m in history['train']], label='train')
axes[1].plot([m['phase_acc'] for m in history['val']], label='val')
axes[1].set_title('Phase accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

fig.tight_layout()
fig


In [ ]:
# Evaluate on held-out graphs
model.eval()
test_stats = run_epoch(test_loader, training=False)
print('Test metrics:', json.dumps(test_stats, indent=2))


In [ ]:
# Visualise a few predictions with Tc intervals
import pandas as pd
from torch_geometric.utils import to_networkx
import networkx as nx

examples = []
all_batches = list(test_loader)
with torch.no_grad():
    for batch in all_batches:
        batch = batch.to(device)
        probs, lower, upper, tc_mean = model.predict_with_interval(batch)
        phases = probs.argmax(dim=-1).cpu()
        for i in range(batch.num_graphs):
            examples.append({
                'pred_phase': int(phases[i].item()),
                'phase_prob_ordered': float(probs[i,0].item()),
                'phase_prob_disordered': float(probs[i,1].item()),
                'true_phase': int(batch.y[i].item()),
                'true_temp': float(batch.temperature[i].item()),
                'pred_temp': float(batch.temperature[i].item()),
                'tc_lower': float(lower[i].cpu().item()),
                'tc_upper': float(upper[i].cpu().item()),
                'tc_mean': float(tc_mean[i].cpu().item()),
                'true_tc': float(batch.tc_target[i].item()),
            })

import pandas as pd
df_examples = pd.DataFrame(examples)
df_examples.head()


In [ ]:
# Scatter plot of Tc intervals
true_tc = np.array([ex['true_tc'] for ex in examples])
pred_mean = np.array([ex['tc_mean'] for ex in examples])
lower = np.array([ex['tc_lower'] for ex in examples])
upper = np.array([ex['tc_upper'] for ex in examples])

plt.figure(figsize=(6,4))
plt.errorbar(true_tc, pred_mean, yerr=[pred_mean - lower, upper - pred_mean], fmt='o', capsize=4)
plt.axline((critical_temperature, critical_temperature), slope=1, color='gray', linestyle='--', label='y=x')
plt.xlabel('True Tc')
plt.ylabel('Predicted Tc with 95% interval')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Visualise a single lattice graph with phase probability shading
sample_graph = test_graphs[0]
sample_graph.batch = torch.zeros(sample_graph.num_nodes, dtype=torch.long)
probs, lower, upper, tc_mean = model.predict_with_interval(sample_graph.to(device))
phase_prob = probs[0,1].item()  # probability of disordered phase

G = to_networkx(sample_graph, to_undirected=True)
pos = {i: (i % lattice_shape[1], i // lattice_shape[1]) for i in range(sample_graph.num_nodes)}

plt.figure(figsize=(5,5))
nx.draw(G, pos=pos, node_size=120, node_color='skyblue', edge_color='lightgray')
plt.title(f"Pred P(disordered)={phase_prob:.2f}, Tc~{tc_mean.item():.2f}")
plt.axis('off')
plt.show()
